## ***ASG Airlines: End-to-End Data Engineering Case Study***


#### **1. Data Ingestion**

The data from the excel file is loaded using Python and Pandas.


In [2]:
import pandas as pd

file="UseCase - Airlines.xlsx"

flights=pd.read_excel(file, sheet_name="flights")
bookings=pd.read_excel(file, sheet_name="bookings")
payments=pd.read_excel(file, sheet_name="payments")
passengers=pd.read_excel(file, sheet_name="passengers")

print("Flights:",flights.shape)
print("Bookings:",bookings.shape)
print("Payments:",payments.shape)
print("Passengers:",passengers.shape)

Flights: (1020, 7)
Bookings: (1000, 9)
Payments: (1000, 4)
Passengers: (1039, 9)


Error handling

In [53]:
try:
    flights = pd.read_excel(file, sheet_name="flights")
    bookings = pd.read_excel(file, sheet_name="bookings")
    payments = pd.read_excel(file, sheet_name="payments")
    passengers = pd.read_excel(file, sheet_name="passengers")

    print("Data loaded successfully.")

except Exception as e:
    print("Error while loading the data:", e)

Data loaded successfully.


**Data Quality Checks**

**Missing values**

In [3]:
def check_missing(df, name):
    print(name)
    print(df.isnull().sum())
    print()
check_missing(flights,"Flights")
check_missing(bookings,"Bookings")
check_missing(payments,"Payments")
check_missing(passengers,"Passengers")

Flights
flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

Bookings
booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64

Payments
payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64

Passengers
passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64



**Duplicate records**

In [4]:
def check_duplicates(df,name):
    print(name,"duplicates:", df.duplicated().sum())
check_duplicates(flights,"Flights")
check_duplicates(bookings,"Bookings")
check_duplicates(payments,"Payments")
check_duplicates(passengers,"Passengers")

Flights duplicates: 15
Bookings duplicates: 0
Payments duplicates: 0
Passengers duplicates: 0


**Checking all airline values**

In [10]:
print("Airline values:")
print(flights["airline"].value_counts(dropna=False))

Airline values:
airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64


**Booking status check**

In [11]:
print("Booking status values:")
print(bookings["status"].value_counts(dropna=False))

Booking status values:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64


**Overall invalid values**

In [7]:
print("UNKNOWN airlines:", flights["airline"].eq("UNKNOWN").sum())

print("INVALID payment amounts:",
      payments["amount"].astype(str).str.upper().eq("INVALID").sum())

print("INVALID booking statuses:",
      bookings["status"].astype(str).str.upper().eq("INVALID").sum())

UNKNOWN airlines: 31
INVALID payment amounts: 30
INVALID booking statuses: 30


**Flight duration check**

In [8]:
flights["calculated_duration"] = (
    flights["arrival_time"] - flights["departure_time"]
)

print("Negative durations:",
      (flights["calculated_duration"].dt.total_seconds() < 0).sum())

print("Overnight flights:",
      (flights["arrival_time"].dt.date >
       flights["departure_time"].dt.date).sum())

Negative durations: 1
Overnight flights: 124


**Check duplicate passenger IDs**

In [9]:
print("Duplicate passenger IDs:",
      passengers["passenger_id"].duplicated().sum())

Duplicate passenger IDs: 39


The data quality check revealed duplicates, missing & invalid values, and some flight timing issues. These issues should be fixed or removed

#### **2. Data Transformation and Cleaning**

**Handling dupliacted records**

In [12]:
flights =flights.drop_duplicates()

print("Flights after removing duplicates:", flights.shape)

Flights after removing duplicates: (1005, 8)


Removed duplicate flight records to ensure that the same flight was not counted more than once

**Handling missing values**

In [13]:
flights["airline"]=flights["airline"].fillna("Unknown")
bookings["status"]= bookings["status"].fillna("Unknown")
passengers["last_name"]= passengers["last_name"].fillna("Unknown")
payments["amount"] = pd.to_numeric(
    payments["amount"], errors="coerce"
)
payments["amount"] =payments["amount"].fillna(0)

Missing categorical values were replaced with Unknown while missing payment amounts were converted to numeric and set to 0 so they could be processed correctly

**Handling invalid records**

In [14]:
bookings["status"] = bookings["status"].replace(
    "INVALID", "Unknown"
)

The INVALID status does not represent a valid booking state so it  changed to Unknown instead of treating it as a real status

In [15]:
payments["amount"] = pd.to_numeric(
    payments["amount"], errors="coerce"
)
payments["amount"] = payments["amount"].fillna(0)

Non-numeric values are replaced with 0 so that payment calculations could be performed correctly

In [28]:
passengers["aadhaar_id"] =passengers["aadhaar_id"].astype(str)

Aadhaar IDs were converted to text so that they would not be treated as numerical values during the processing

In [29]:
flights["departure_time"]=pd.to_datetime(flights["departure_time"])
flights["arrival_time"]=pd.to_datetime(flights["arrival_time"])
bookings["booking_date"]=pd.to_datetime(bookings["booking_date"])
passengers["date_of_birth"]=pd.to_datetime(passengers["date_of_birth"])

For the negative flight duration, first calculate the time duartion from the timestamp and then find the invalid records. But since the data has only one negative-duration record, it can be removed


In [27]:
flights["duration_minutes"] = (
    flights["arrival_time"] - flights["departure_time"]
).dt.total_seconds() / 60

flights =flights[flights["duration_minutes"] >= 0]

Now we also have a reliable numeric duration column "duration_minutes" that can be used for the trnsformtaion stage

**Stadardizing column values**

In [32]:
flights["airline"]=flights["airline"].str.strip()
flights["source"]=flights["source"].str.strip().str.upper()
flights["destination"]=flights["destination"].str.strip().str.upper()

bookings["status"]=bookings["status"].str.strip().str.upper()

passengers["gender"]=passengers["gender"].str.strip().str.upper()

**PII masking**

Aadhaar ID and passport number are the most sensitive information in the dataset. Since they are not required for the KPIs or Power BI dashboard they could be removed. But I have chosen to mask them to protect the sensitive information.

In [35]:

passengers["aadhaar_id"] = (
    passengers["aadhaar_id"].astype(str).str[-4:].str.rjust(12, "*")
)
bookings["passport_number"] = "********"

**Validation**

In [52]:
print("Negative durations:", (flights["duration_minutes"] < 0).sum())
print("Duplicate flights:", flights.duplicated().sum())
print("Missing airline:", flights["airline"].isnull().sum())
print("Missing booking status:", bookings["status"].isnull().sum())
print("Missing payment amount:", payments["amount"].isnull().sum())

Negative durations: 0
Duplicate flights: 0
Missing airline: 0
Missing booking status: 0
Missing payment amount: 0


#### **3. Data Modelling, Storage and Business KPIs**

**Flight duration avergae :** This gives the average duration of all valid flights

In [36]:
average_duration =flights["duration_minutes"].mean()
print("Average Flight Duration:",
      round(average_duration, 2), "minutes")

Average Flight Duration: 164.49 minutes


**Route-wise traffic :** this creates a route column &  count the flights for each route.

In [39]:
flights["route"] =(
    flights["source"]+"-" +flights["destination"]
)
route_traffic = flights["route"].value_counts()

print("Route-wise traffic:")
print(route_traffic)

Route-wise traffic:
route
BOM-CCU    90
CCU-DEL    72
MAA-BLR    65
BLR-BOM    60
HYD-MAA    57
DEL-HYD    54
HYD-DEL    42
BOM-DEL    39
CCU-BOM    33
DEL-BLR    29
DEL-BOM    28
BOM-MAA    27
MAA-DEL    26
HYD-BLR    26
HYD-CCU    26
BOM-HYD    26
HYD-BOM    26
DEL-CCU    26
MAA-CCU    24
CCU-MAA    24
DEL-MAA    23
BOM-BLR    23
CCU-HYD    21
MAA-HYD    21
BLR-CCU    21
MAA-BOM    21
CCU-BLR    20
BLR-HYD    19
BLR-DEL    19
BLR-MAA    16
Name: count, dtype: int64


**Delays/ Anomalies :** Since we already removed the negative-duration record during cleaning the expected result should be: 0

In [40]:
anomalies = flights[
    flights["duration_minutes"]<=0
]
print("Number of anomalies:", len(anomalies))

Number of anomalies: 0


**Distribution of the flights**

In [43]:
airline_distribution = flights[
    "airline"
].value_counts()
print("Flights by Airline:")
print(airline_distribution)

Flights by Airline:
airline
IndiGo       249
SpiceJet     235
Air India    233
Vistara      218
Unknown       39
UNKNOWN       30
Name: count, dtype: int64


**Basic KPIs**

In [44]:
print("Total Flights:",len(flights))
print("Total Bookings:",len(bookings))
print("Total Payment Amount:",
      round(payments["amount"].sum(), 2))
print("Average Payment Amount:",
      round(payments["amount"].mean(), 2))
booking_status = bookings["status"].value_counts()
print("Booking Status Distribution:")
print(booking_status)

Total Flights: 1004
Total Bookings: 1000
Total Payment Amount: 7385142.98
Average Payment Amount: 7385.14
Booking Status Distribution:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       75
Name: count, dtype: int64


The cleaned data was used to calculate key business KPIs such as average flight duration, route-wise traffic, flight anomalies, and airline-wise flight distribution. Additional metrics such as total flights, bookings, and payment amounts were also calculated to support further analysis and reporting.

#### **4. Power BI Dashboard**

**KPI table**

In [45]:
kpi_summary=pd.DataFrame({
    "KPI": [
        "Total Flights",
        "Average Flight Duration",
        "Minimum Flight Duration",
        "Maximum Flight Duration",
        "Total Bookings",
        "Total Payment Amount"
    ],
    "Value":[
        len(flights),
        round(flights["duration_minutes"].mean(), 2),
        round(flights["duration_minutes"].min(), 2),
        round(flights["duration_minutes"].max(), 2),
        len(bookings),
        round(payments["amount"].sum(), 2)
    ]
})
kpi_summary

,KPI,Value
0,Total Flights,1004.00
1,Average Flight Duration,164.49
2,Minimum Flight Duration,30.00
3,Maximum Flight Duration,300.00
4,Total Bookings,1000.00
5,Total Payment Amount,7385142.98


**Route table**

In [46]:
route_traffic=flights.groupby(
    ["source","destination"]
).size().reset_index(name="flight_count")
route_traffic["route"]=(
    route_traffic["source"] + "-" +
    route_traffic["destination"]
)

route_traffic

,source,destination,flight_count,route
0,BLR,BOM,60,BLR-BOM
1,BLR,CCU,21,BLR-CCU
2,BLR,DEL,19,BLR-DEL
3,BLR,HYD,19,BLR-HYD
4,BLR,MAA,16,BLR-MAA
5,BOM,BLR,23,BOM-BLR
6,BOM,CCU,90,BOM-CCU
7,BOM,DEL,39,BOM-DEL
8,BOM,HYD,26,BOM-HYD
9,BOM,MAA,27,BOM-MAA


**Airline table**

In [47]:
airline_trends =flights.groupby(
    "airline"
).size().reset_index(name="flight_count")
airline_trends

,airline,flight_count
0,Air India,233
1,IndiGo,249
2,SpiceJet,235
3,UNKNOWN,30
4,Unknown,39
5,Vistara,218


**Duration taable**

In [48]:
duration_analysis= flights[
    ["flight_id", "airline", "source",
     "destination", "duration_minutes"]
]

duration_analysis.head()

,flight_id,airline,source,destination,duration_minutes
0,SJ010,SpiceJet,CCU,MAA,174.0
1,AI155,Air India,BOM,CCU,108.0
2,UK094,Vistara,BOM,CCU,105.0
3,AI245,Air India,BOM,CCU,156.0
4,AI192,Air India,MAA,BOM,299.0


**Anamoly table**

In [49]:
anomaly_analysis = flights[
    flights["duration_minutes"] > 300
][
    ["flight_id", "airline", "source",
     "destination", "duration_minutes"]
]
anomaly_analysis

,flight_id,airline,source,destination,duration_minutes


Exporting cleaned datasets

In [54]:

flights.to_csv("output/cleaned_flights.csv", index=False)
bookings.to_csv("output/cleaned_bookings.csv", index=False)
payments.to_csv("output/cleaned_payments.csv", index=False)
passengers.to_csv("output/cleaned_passengers.csv", index=False)

Exporting KPI tables

In [51]:
kpi_summary.to_csv("output/kpi_summary.csv", index=False)
route_traffic.to_csv("output/route_traffic.csv", index=False)
airline_trends.to_csv("output/airline_trends.csv", index=False)
duration_analysis.to_csv("output/duration_analysis.csv", index=False)
anomaly_analysis.to_csv("output/anomaly_analysis.csv", index=False)